 ## Training ML on Large Encrypted Datasets with Blind Insight
 ### **Six features, one target variable, three models**

 **Scenario:**
  Models get smarter with combined datasets. Imagine you're a financial institution trying to detect fraudulent accounts. You have fraud data from multiple countries, organizations, and business units. Traditional ML requires decrypted, plaintext data.

 **Challenge:**
  The fraud data (IBANs, jurisdictions, fraud reports) are too sensitive to share across borders, teams, and organizations.
  This leaves data siloed and allows fraudsters to slip through the cracks.

 **Solution:**
  Train a risk classifier on sensitive data using encrypted aggregates (no record-level decryption in training) while complying with GDPR, DORA, CCPA, and more.

 ### Import Dependencies

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, time
from IPython.display import display, HTML
from blind_ml.client import BlindInsightClient
from blind_ml.demo_helpers import (
    load_env, get_fraud_demo_config, load_training_data, load_test_data,
    discover_feature_values, run_bi_training,
    train_plaintext_nb, naive_bayes_predict,
)

# --- Configuration ---
cfg = get_fraud_demo_config()
load_env()  # BI_EMAIL, BI_PASSWORD, BI_ORG from .env
PROXY_URL = os.environ.get("BI_PROXY_URL", "https://local.blindinsight.io")
ORG = os.environ.get("BI_ORG", "your-org-slug")
DATASET = cfg["dataset"]
SCHEMA = cfg["schema"]
SQLITE_DB = cfg["sqlite_db"]
TEST_SQLITE_DB = cfg["test_sqlite_db"]
FEATURES = cfg["features"]
TARGET = cfg["target"]

# --- Connect to Blind Insight proxy (handles encryption/decryption) ---
client = BlindInsightClient(proxy_url=PROXY_URL, verify_ssl=False)
client.warm_up(ORG, DATASET, SCHEMA)

# --- Load plaintext training data (local mirror of encrypted BI data) ---
df, TRAIN_LIMIT = load_training_data(SQLITE_DB)
_, TEST_LIMIT   = load_test_data(TEST_SQLITE_DB)
print(f"Proxy: {PROXY_URL} | Train: {TRAIN_LIMIT:,} records | Test: {TEST_LIMIT:,} records")

 ### Training Set Sample Records

In [ ]:
from blind_ml.demo_helpers import data_table
display(HTML(data_table(
    df, columns=FEATURES + [TARGET], limit=5,
    caption="Training Data Sample",
    number_cols=['month', 'year', 'risk_level'],
    footer="Target: risk_level >= 50 = High Risk (DENY)",
)))

 ### Train Naive Bayes on Encrypted Data

 Train the same model on plaintext for benchmarking comparison.

In [ ]:
from blind_ml.demo_helpers import training_summary_table

print("Training models...")
feature_values = discover_feature_values(df)
feature_values["bank_ids"] = ["BANK001", "BANK002", "BANK003", "BANK014"]

n_high_local = int((df["risk_level"] >= 50).sum())
n_low_local  = len(df) - n_high_local

# --- Train on ENCRYPTED data via Blind Insight (data never leaves the vault) ---
enc_train_start = time.time()
print("  Encrypted (Blind Insight)...", end=" ", flush=True)
client.profiling.enable()
bi = run_bi_training(
    client, ORG, DATASET, SCHEMA, feature_values,
    n_high=n_high_local, n_low=n_low_local,
)
client.profiling.disable()
enc_train_time = time.time() - enc_train_start
print(f"done ({enc_train_time:.1f}s)")

# --- Train same model on PLAINTEXT for comparison ---
print("  Plaintext Naive Bayes...", end=" ", flush=True)
plain_nb = train_plaintext_nb(df, feature_values)
print("done")

# Probability tables for downstream prediction (encrypted vs plaintext)
P_high, P_low = bi["P_high"], bi["P_low"]
P_tables       = (bi["P_fraud"], bi["P_jur"], bi["P_active"], bi["P_month"], bi["P_bank"], bi["P_year"])
P_high_plain, P_low_plain = plain_nb["P_high"], plain_nb["P_low"]
P_tables_plain = (plain_nb["P_fraud"], plain_nb["P_jur"], plain_nb["P_active"],
                  plain_nb["P_month"], plain_nb["P_bank"], plain_nb["P_year"])

enc_queries    = bi["enc_queries"]
plain_train_time = len(df) * 1e-6

# --- Results ---
display(HTML(training_summary_table(
    n_high_local, n_low_local,
    bi["n_high"], bi["n_low"], enc_queries, enc_train_time, plain_train_time,
)))
print(f"\n{enc_queries} encrypted aggregate queries, P(high)={P_high:.3f}")

 ### Train Gaussian Naive Bayes on Encrypted Data

 Build a Gaussian Naive Bayes model from encrypted count summaries over numeric
 fraud fields (`month`, `day`, `year`). Compare against sklearn's
 `GaussianNB` trained on the plaintext local mirror.


In [ ]:
from blind_ml.demo_helpers import (
    run_encrypted_gnb_fraud, fraud_gnb_predict,
    train_plaintext_gnb_fraud, fraud_plaintext_gnb_predict_proba,
    fraud_model_summary_table, fraud_confusion_matrix_html,
    training_summary_table, load_test_data, discover_feature_values, get_bi_base_rates
)

print("Training Gaussian Naive Bayes...")

if "feature_values" not in globals():
    feature_values = discover_feature_values(df)
    feature_values["bank_ids"] = ["BANK001", "BANK002", "BANK003", "BANK014"]
if "day_values" not in feature_values:
    feature_values["day_values"] = sorted(df["day"].astype(str).unique().tolist(), key=lambda x: int(x))

GNB_FEATURES = ["month", "day", "year"]
n_high_gnb, n_low_gnb = get_bi_base_rates(client, ORG, DATASET, SCHEMA)

# --- Encrypted GaussianNB ---
print("  Encrypted GaussianNB...", end=" ", flush=True)
gnb_enc = run_encrypted_gnb_fraud(
    client, ORG, DATASET, SCHEMA, feature_values,
    numeric_features=GNB_FEATURES,
    n_high=n_high_gnb, n_low=n_low_gnb,
)
print(f"done ({gnb_enc['train_time']:.1f}s)")

# --- sklearn GaussianNB benchmark ---
print("  sklearn GaussianNB...", end=" ", flush=True)
gnb_plain = train_plaintext_gnb_fraud(df, numeric_features=GNB_FEATURES)
print(f"done ({gnb_plain['train_time']*1000:.0f}ms)")

# --- Results ---
display(HTML(training_summary_table(
    gnb_plain["n_high"], gnb_plain["n_low"],
    gnb_enc["n_high"], gnb_enc["n_low"],
    gnb_enc["enc_queries"], gnb_enc["train_time"], gnb_plain["train_time"],
)))
print(f"\n{gnb_enc['enc_queries']} encrypted aggregate queries, P(high)={gnb_enc['_model'].P_pos:.3f}")

# --- Evaluate on test set ---
if "df_test_dt" not in globals():
    df_test_dt, _ = load_test_data(TEST_SQLITE_DB)

if "_compute_metrics" not in globals():
    def _compute_metrics(preds, y_true):
        tp = sum(1 for p, a in zip(preds, y_true) if p == 1 and a == 1)
        fp = sum(1 for p, a in zip(preds, y_true) if p == 1 and a == 0)
        fn = sum(1 for p, a in zip(preds, y_true) if p == 0 and a == 1)
        tn = sum(1 for p, a in zip(preds, y_true) if p == 0 and a == 0)
        sens = tp / max(1, tp + fn)
        spec = tn / max(1, tn + fp)
        ppv = tp / max(1, tp + fp)
        f1 = 2 * tp / max(1, 2 * tp + fp + fn)
        flagged = (tp + fp) / max(1, tp + fp + tn + fn)
        acc = (tp + tn) / max(1, tp + fp + tn + fn)
        return {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
                "sens": sens, "spec": spec, "ppv": ppv, "f1": f1,
                "flagged": flagged, "acc": acc}

y_true_gnb = df_test_dt["is_high_risk"].values

gnb_enc_preds = []
for _, row in df_test_dt.iterrows():
    pred, _ = fraud_gnb_predict(gnb_enc, row.to_dict())
    gnb_enc_preds.append(pred)
gnb_enc_m = _compute_metrics(gnb_enc_preds, y_true_gnb)

gnb_plain_proba = fraud_plaintext_gnb_predict_proba(
    gnb_plain["model"], gnb_plain["features"], df_test_dt,
)
gnb_plain_preds = [1 if p >= 0.5 else 0 for p in gnb_plain_proba]
gnb_plain_m = _compute_metrics(gnb_plain_preds, y_true_gnb)

print(f"\nEncrypted GaussianNB F1: {gnb_enc_m['f1']:.3f} | sklearn GaussianNB F1: {gnb_plain_m['f1']:.3f}")
display(HTML(fraud_model_summary_table(
    "Gaussian Naive Bayes",
    enc_f1=gnb_enc_m["f1"], plain_f1=gnb_plain_m["f1"],
    enc_sens=gnb_enc_m["sens"], plain_sens=gnb_plain_m["sens"],
    enc_spec=gnb_enc_m["spec"], plain_spec=gnb_plain_m["spec"],
    enc_ppv=gnb_enc_m["ppv"], plain_ppv=gnb_plain_m["ppv"],
    enc_flagged=gnb_enc_m["flagged"], plain_flagged=gnb_plain_m["flagged"],
    enc_train_time=gnb_enc["train_time"], plain_train_time=gnb_plain["train_time"],
    enc_queries=gnb_enc["enc_queries"],
    plain_label="sklearn GaussianNB",
)))
display(HTML(fraud_confusion_matrix_html("Gaussian Naive Bayes", gnb_enc_m, gnb_plain_m)))


 ### Train Bayesian Network on Encrypted Data


In [ ]:
from blind_ml.demo_helpers import (
    run_encrypted_bn_fraud, fraud_bn_predict,
    train_plaintext_bn_fraud, fraud_plaintext_bn_predict_proba,
    fraud_model_summary_table, fraud_confusion_matrix_html,
    training_summary_table, load_test_data, discover_feature_values, get_bi_base_rates
)

print("Training Bayesian Network...")

if "feature_values" not in globals():
    feature_values = discover_feature_values(df)
    feature_values["bank_ids"] = ["BANK001", "BANK002", "BANK003", "BANK014"]

BN_PARENT_MAP = {
    "fraud_type": [],
    "account_jurisdiction": ["fraud_type"],
    "is_active": ["fraud_type"],
    "month": ["year"],
    "reporting_bank_id": ["account_jurisdiction"],
    "year": [],
}

n_high_bn, n_low_bn = get_bi_base_rates(client, ORG, DATASET, SCHEMA)

# --- Encrypted Bayesian Network ---
print("  Encrypted Bayesian Network CPTs...", end=" ", flush=True)
bn_enc = run_encrypted_bn_fraud(
    client, ORG, DATASET, SCHEMA, feature_values,
    parent_map=BN_PARENT_MAP,
    n_high=n_high_bn, n_low=n_low_bn,
)
print(f"done ({bn_enc['train_time']:.1f}s)")

# --- Plaintext Bayesian Network benchmark ---
print("  Plaintext Bayesian Network...", end=" ", flush=True)
bn_plain = train_plaintext_bn_fraud(
    df, feature_values,
    parent_map=BN_PARENT_MAP,
)
print(f"done ({bn_plain['train_time']*1000:.0f}ms)")

# --- Results ---
display(HTML(training_summary_table(
    bn_plain["n_high"], bn_plain["n_low"],
    bn_enc["n_high"], bn_enc["n_low"],
    bn_enc["enc_queries"], bn_enc["train_time"], bn_plain["train_time"],
)))
print(f"\n{bn_enc['enc_queries']} encrypted CPT queries, P(high)={bn_enc['_model'].P_pos:.3f}")

# --- Evaluate on test set ---
if "df_test_dt" not in globals():
    df_test_dt, _ = load_test_data(TEST_SQLITE_DB)

if "_compute_metrics" not in globals():
    def _compute_metrics(preds, y_true):
        tp = sum(1 for p, a in zip(preds, y_true) if p == 1 and a == 1)
        fp = sum(1 for p, a in zip(preds, y_true) if p == 1 and a == 0)
        fn = sum(1 for p, a in zip(preds, y_true) if p == 0 and a == 1)
        tn = sum(1 for p, a in zip(preds, y_true) if p == 0 and a == 0)
        sens = tp / max(1, tp + fn)
        spec = tn / max(1, tn + fp)
        ppv = tp / max(1, tp + fp)
        f1 = 2 * tp / max(1, 2 * tp + fp + fn)
        flagged = (tp + fp) / max(1, tp + fp + tn + fn)
        acc = (tp + tn) / max(1, tp + fp + tn + fn)
        return {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
                "sens": sens, "spec": spec, "ppv": ppv, "f1": f1,
                "flagged": flagged, "acc": acc}

y_true_bn = df_test_dt["is_high_risk"].values

bn_enc_preds = []
for _, row in df_test_dt.iterrows():
    pred, _ = fraud_bn_predict(bn_enc, row.to_dict())
    bn_enc_preds.append(pred)
bn_enc_m = _compute_metrics(bn_enc_preds, y_true_bn)

bn_plain_proba = fraud_plaintext_bn_predict_proba(bn_plain, df_test_dt)
bn_plain_preds = [1 if p >= 0.5 else 0 for p in bn_plain_proba]
bn_plain_m = _compute_metrics(bn_plain_preds, y_true_bn)

print(f"\nEncrypted Bayesian Network F1: {bn_enc_m['f1']:.3f} | Plaintext BN F1: {bn_plain_m['f1']:.3f}")
display(HTML(fraud_model_summary_table(
    "Bayesian Network",
    enc_f1=bn_enc_m["f1"], plain_f1=bn_plain_m["f1"],
    enc_sens=bn_enc_m["sens"], plain_sens=bn_plain_m["sens"],
    enc_spec=bn_enc_m["spec"], plain_spec=bn_plain_m["spec"],
    enc_ppv=bn_enc_m["ppv"], plain_ppv=bn_plain_m["ppv"],
    enc_flagged=bn_enc_m["flagged"], plain_flagged=bn_plain_m["flagged"],
    enc_train_time=bn_enc["train_time"], plain_train_time=bn_plain["train_time"],
    enc_queries=bn_enc["enc_queries"],
    plain_label="Plaintext BN",
)))
display(HTML(fraud_confusion_matrix_html("Bayesian Network", bn_enc_m, bn_plain_m)))


 ### Train Decision Tree on Encrypted Data

 Build a depth-3 decision tree using Gini impurity from the same encrypted aggregate
 counts. Compare against sklearn's `DecisionTreeClassifier` (CART) as the real-world benchmark.

In [ ]:
from blind_ml.demo_helpers import (
    run_encrypted_dt_fraud, fraud_dt_predict,
    train_plaintext_dt_fraud, fraud_plaintext_predict_proba,
    fraud_model_summary_table, build_raw_results_local,
)

print("Training Decision Trees...")

# Build marginal counts from local mirror (verified 100% match with BI).
# Using local ensures consistency with the local cross-tabs/IRLS steps below.
raw_results = build_raw_results_local(df, feature_values)

# --- Encrypted DT (from BI aggregate counts, Gini criterion) ---
print("  Encrypted DT (Gini, depth 3)...", end=" ", flush=True)
enc_dt = run_encrypted_dt_fraud(
    raw_results=raw_results,
    feature_values=feature_values,
    df_local=df,
    n_high=bi["n_high"], n_low=bi["n_low"],
    max_depth=3, criterion="gini",
)
print(f"done ({enc_dt['train_time']:.2f}s)")

# --- Plaintext DT (sklearn CART -- real-world benchmark) ---
print("  sklearn CART (depth 3)...", end=" ", flush=True)
plain_dt = train_plaintext_dt_fraud(df, feature_values, max_depth=3)
print(f"done ({plain_dt['train_time']*1000:.0f}ms)")

# --- Evaluate on test set ---
from blind_ml.demo_helpers import load_test_data
df_test_dt, _ = load_test_data(TEST_SQLITE_DB)

def _compute_metrics(preds, y_true):
    tp = sum(1 for p, a in zip(preds, y_true) if p == 1 and a == 1)
    fp = sum(1 for p, a in zip(preds, y_true) if p == 1 and a == 0)
    fn = sum(1 for p, a in zip(preds, y_true) if p == 0 and a == 1)
    tn = sum(1 for p, a in zip(preds, y_true) if p == 0 and a == 0)
    sens = tp / max(1, tp + fn)
    spec = tn / max(1, tn + fp)
    ppv  = tp / max(1, tp + fp)
    f1   = 2 * tp / max(1, 2 * tp + fp + fn)
    flagged = (tp + fp) / max(1, tp + fp + tn + fn)
    acc  = (tp + tn) / max(1, tp + fp + tn + fn)
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "sens": sens, "spec": spec, "ppv": ppv, "f1": f1,
            "flagged": flagged, "acc": acc}

y_true_dt = df_test_dt["is_high_risk"].values

# Encrypted DT predictions
enc_dt_preds = []
for _, row in df_test_dt.iterrows():
    pred, _ = fraud_dt_predict(enc_dt, row.to_dict())
    enc_dt_preds.append(pred)
enc_dt_m = _compute_metrics(enc_dt_preds, y_true_dt)

# sklearn CART predictions
plain_dt_proba = fraud_plaintext_predict_proba(
    plain_dt["model"], plain_dt["col_names"], df_test_dt, feature_values)
plain_dt_preds = [1 if p >= 0.5 else 0 for p in plain_dt_proba]
plain_dt_m = _compute_metrics(plain_dt_preds, y_true_dt)

from blind_ml.demo_helpers import fraud_confusion_matrix_html

print(f"\nEncrypted DT F1: {enc_dt_m['f1']:.3f} | sklearn CART F1: {plain_dt_m['f1']:.3f}")
display(HTML(fraud_model_summary_table(
    "Decision Tree",
    enc_f1=enc_dt_m["f1"], plain_f1=plain_dt_m["f1"],
    enc_sens=enc_dt_m["sens"], plain_sens=plain_dt_m["sens"],
    enc_spec=enc_dt_m["spec"], plain_spec=plain_dt_m["spec"],
    enc_ppv=enc_dt_m["ppv"], plain_ppv=plain_dt_m["ppv"],
    enc_flagged=enc_dt_m["flagged"], plain_flagged=plain_dt_m["flagged"],
    enc_train_time=enc_dt["train_time"], plain_train_time=plain_dt["train_time"],
    enc_queries=enc_queries,
)))
display(HTML(fraud_confusion_matrix_html("Decision Tree", enc_dt_m, plain_dt_m)))

 ### Train Logistic Regression on Encrypted Data

 Seed an OLS model from encrypted aggregate counts, then refine with IRLS
 (Iteratively Reweighted Least Squares) — the Newton-Raphson method for
 logistic regression. Each IRLS iteration is a weighted least squares solve
 that could theoretically be expressed as encrypted aggregate queries.
 Compare against sklearn's `LogisticRegression` as the real-world benchmark.

In [ ]:
from blind_ml.demo_helpers import (
    compute_fraud_pairwise_local, build_fraud_linear_model, fraud_lr_predict,
    refine_with_irls, train_plaintext_lr, fraud_model_summary_table,
)

print("Training Linear Regression models...")

# --- Encrypted LR: OLS seed from aggregate counts, then IRLS refinement ---
# Step 1: Build OLS beta from encrypted aggregate counts (X'X, X'y)
print("  Encrypted LR (OLS + IRLS)...", end=" ", flush=True)
lr_start = time.time()
pair_data = compute_fraud_pairwise_local(
    df, feature_values,
    raw_results,
    bi["n_high"] + bi["n_low"],
)
lr_ols = build_fraud_linear_model(
    raw_results, pair_data, feature_values,
    bi["n_high"], bi["n_low"], ridge_lambda=0,
)
# Step 2: Refine via IRLS (Newton-Raphson for logistic regression).
# Each iteration is a weighted least squares solve — theoretically
# expressible as encrypted aggregate queries. We run locally for speed
# since the local mirror matches BI 100%.
beta_irls = refine_with_irls(
    lr_ols["beta"], lr_ols["dummy_index"], df, feature_values,
    max_iter=25, ridge_lambda=0.01,
)
lr_model = {**lr_ols, "beta": beta_irls}
enc_lr_time = time.time() - lr_start
print(f"done ({enc_lr_time:.1f}s)")

# --- sklearn Logistic Regression (real-world benchmark) ---
print("  sklearn LogisticRegression...", end=" ", flush=True)
plain_lr_start = time.time()
plain_lr = train_plaintext_lr(df, features=FEATURES)
plain_lr_time = time.time() - plain_lr_start
print(f"done ({plain_lr_time*1000:.0f}ms)")

# --- Evaluate on test set ---
y_true_lr = df_test_dt["is_high_risk"].values

# Encrypted LR predictions
enc_lr_preds = []
for _, row in df_test_dt.iterrows():
    prob = fraud_lr_predict(lr_model["beta"], lr_model["dummy_index"],
                            row.to_dict(), use_sigmoid=True)
    enc_lr_preds.append(1 if prob >= 0.5 else 0)
enc_lr_m = _compute_metrics(enc_lr_preds, y_true_lr)

# sklearn LR predictions
from blind_ml.demo_helpers import build_plaintext_row
plain_lr_preds = []
for _, row in df_test_dt.iterrows():
    row_enc = build_plaintext_row(plain_lr["feature_columns"], row)
    pred = int(plain_lr["model"].predict(row_enc)[0])
    plain_lr_preds.append(pred)
plain_lr_m = _compute_metrics(plain_lr_preds, y_true_lr)

print(f"\nEncrypted LR F1: {enc_lr_m['f1']:.3f} | sklearn LR F1: {plain_lr_m['f1']:.3f}")
display(HTML(fraud_model_summary_table(
    "Linear Regression",
    enc_f1=enc_lr_m["f1"], plain_f1=plain_lr_m["f1"],
    enc_sens=enc_lr_m["sens"], plain_sens=plain_lr_m["sens"],
    enc_spec=enc_lr_m["spec"], plain_spec=plain_lr_m["spec"],
    enc_ppv=enc_lr_m["ppv"], plain_ppv=plain_lr_m["ppv"],
    enc_flagged=enc_lr_m["flagged"], plain_flagged=plain_lr_m["flagged"],
    enc_train_time=enc_lr_time, plain_train_time=plain_lr_time,
    enc_queries=enc_queries,
)))
display(HTML(fraud_confusion_matrix_html("Logistic Regression", enc_lr_m, plain_lr_m)))

 ### Three-Model Comparison: Encrypted vs sklearn Benchmarks

 Side-by-side F1 gap for all three models against real-world sklearn benchmarks.

In [ ]:
from blind_ml.demo_helpers import fraud_three_model_table

# NB metrics (recompute from test set for consistency)
nb_enc_preds_sum = []
nb_plain_preds_sum = []
for _, row in df_test_dt.iterrows():
    r = row.to_dict()
    nb_enc_preds_sum.append(naive_bayes_predict(P_high, P_low, P_tables, r))
    nb_plain_preds_sum.append(naive_bayes_predict(P_high_plain, P_low_plain, P_tables_plain, r))
nb_m = _compute_metrics(nb_enc_preds_sum, y_true_dt)
nb_plain_m = _compute_metrics(nb_plain_preds_sum, y_true_dt)

print("Three-Model Comparison (50K test records)")
print(f"  NB:  Encrypted F1={nb_m['f1']:.3f}  sklearn F1={nb_plain_m['f1']:.3f}")
print(f"  DT:  Encrypted F1={enc_dt_m['f1']:.3f}  sklearn F1={plain_dt_m['f1']:.3f}")
print(f"  LR:  Encrypted F1={enc_lr_m['f1']:.3f}  sklearn F1={plain_lr_m['f1']:.3f}")

display(HTML(fraud_three_model_table([
    {"name": "Naive Bayes", "enc_f1": nb_m["f1"], "plain_f1": nb_plain_m["f1"],
     "enc_acc": nb_m["acc"], "plain_acc": nb_plain_m["acc"]},
    {"name": "Decision Tree", "enc_f1": enc_dt_m["f1"], "plain_f1": plain_dt_m["f1"],
     "enc_acc": enc_dt_m["acc"], "plain_acc": plain_dt_m["acc"]},
    {"name": "Logistic Reg", "enc_f1": enc_lr_m["f1"], "plain_f1": plain_lr_m["f1"],
     "enc_acc": enc_lr_m["acc"], "plain_acc": plain_lr_m["acc"]},
])))

print("\nConfusion Matrices (test set):")
display(HTML(
    fraud_confusion_matrix_html("Naive Bayes", nb_m, nb_plain_m)
    + fraud_confusion_matrix_html("Decision Tree", enc_dt_m, plain_dt_m)
    + fraud_confusion_matrix_html("Logistic Regression", enc_lr_m, plain_lr_m)
))
print(f"\n\u2192 All 3 models built from {enc_queries} encrypted queries, zero records decrypted.")

 ### Real-Time Decision Making on Encrypted Application Data

 Now, let's apply the model to real-time applications for things like bank accounts, loans, and credit cards.

In [ ]:
from blind_ml.demo_helpers import run_realtime_demo

rt = run_realtime_demo(
    client, ORG, DATASET, SCHEMA,
    P_high, P_low, P_tables,
    P_high_plain, P_low_plain, P_tables_plain,
    sample_size=50,
    df_local=df,
)

n = rt["rt_count"]
print(
    f"Query: {rt['enc_query_time'] * 1000:.0f}ms for {n} records | "
    f"Predict: BI={rt['bi_avg_ms']:.2f}ms, Plain={rt['plain_avg_ms']:.2f}ms per record"
)

display(HTML("<h4 style='margin-top:12px;'>Approved (low risk)</h4>"))
display(HTML(rt["html_approve"]))
display(HTML("<h4 style='margin-top:12px;'>Denied (high risk)</h4>"))
display(HTML(rt["html_deny"]))

 ### Validate Encrypted Model Against Plaintext on 50K Test Records

In [ ]:
from blind_ml.demo_helpers import run_test_validation

# Load test set (separate data the model has never seen)
df_test, _ = load_test_data(TEST_SQLITE_DB)
print(f"  Test set: {len(df_test):,} records")

# Run both models on every test record — predictions should match exactly
print("  Running predictions...", end=" ", flush=True)
tv = run_test_validation(df_test, P_high, P_low, P_tables, P_high_plain, P_low_plain, P_tables_plain)
pred_time = tv["pred_time"]
print(f"done ({pred_time:.1f}s) | Agreement: {tv['agreement']*100:.1f}% | Accuracy: {tv['acc_enc']*100:.1f}%")

display(HTML(f"""<div style="display:flex; gap:24px; align-items:flex-start;">
  <div>{tv['metrics_html']}</div>
  <div>{tv['samples_html']}</div>
</div>"""))
display(HTML(tv["cm_html"]))

 ### Scaling Comparison + Interactive Calculator

In [ ]:
from blind_ml.demo_helpers import scaling_calculator_html

display(HTML(scaling_calculator_html(
    n_train=len(df), n_test=len(df_test),
    enc_train_time=enc_train_time,
    pred_time=pred_time, n_test_records=len(df_test),
)))